In [1]:
import scipy.io as sio
import numpy as np
import os
import glob
from matplotlib import pyplot as plt
import re
import random
import json
import pandas as pd

In [2]:
def save_data(save_path, filename, tf):
    if os.path.exists(os.path.join(save_path, filename)):
        return
    else:
        sio.savemat(os.path.join(save_path, filename), tf)


In [3]:
data_path = "/path/to/mae/data/fixed_data_final"

In [4]:
target_files = glob.glob(os.path.join(data_path, "*.mat"))

In [5]:
len(target_files)

54

In [6]:
lco_files = [i for i in target_files if "LCO" in i]
nmc_files = [i for i in target_files if "NMC" in i]
print(len(lco_files))
print(len(nmc_files))

27
27


In [15]:
#---生成mae_train数据---#
expand_factor = 1
mae_train = "/path/to/mae/data/mixup_mae_train/mix_{expand_factor}".format(expand_factor=expand_factor)
os.makedirs(mae_train, exist_ok=True)

file_groups = {}

for file_path in lco_files + nmc_files:
        filename = os.path.basename(file_path)
        
        # Extract battery name and state
        # Pattern: {battery_name}_{state}_wo_{number}.mat
        parts = filename.split('_wo_')[0]
        battery_name, state = parts.split('_', 1)
        
        # Create key for grouping
        group_key = (battery_name, state)
        
        # Add file to group
        if group_key not in file_groups:
            file_groups[group_key] = []
        file_groups[group_key].append(file_path)
        
#-----mixup data generation-----#    
# 定义保存路径
save_path = mae_train


# Generate weights with finer granularity based on expand_factor
step = 1 / expand_factor  # smaller step size for higher granularity
weight = []
for i in range(1, expand_factor):  # more pairs based on expand_factor
    w1 = round(i * step, 3)  # round to 3 decimal places
    w2 = round(1 - w1, 3)
    weight.append((w1, w2))

for (battery, state), files in file_groups.items():
    files = sorted(files)
    if len(files) < 2:
        print(f"Not enough files for {battery}_{state} to perform mixup.")
        continue
    for idx1 in range(len(files)):
        for idx2 in range(idx1 + 1, len(files)):
            stage1 = int(re.search(r'wo_(\d+)', files[idx1]).group(1))
            stage2 = int(re.search(r'wo_(\d+)', files[idx2]).group(1))
            
            tf1 = sio.loadmat(files[idx1])
            filename1 = os.path.basename(files[idx1])
            save_data(save_path, filename1, tf1)
            data1 = tf1['data']
            mah1 = tf1['ah'][0][0]
            
            tf2 = sio.loadmat(files[idx2])
            filename2 = os.path.basename(files[idx2])
            save_data(save_path, filename2, tf2)
            data2 = tf2['data']
            mah2 = tf2['ah'][0][0]
            
            for wt1, wt2 in weight:
                new_data = wt1 * data1 + wt2 * data2
                new_mah = wt1 * mah1 + wt2 * mah2
                new_tf = {'data': new_data, 'ah': np.array([[new_mah]])}
                new_filename = f"{battery}_{state}_wo_{stage1}_w{wt1}_{stage2}_w{wt2}.mat"
                save_data(save_path, new_filename, new_tf)

In [16]:

data_files = os.listdir(mae_train)
len(data_files)

54

In [7]:
total_path = "/path/to/mae/data/lco_unknown/regreesion_data_onlco#4"
# --- 生成 LCO 增强与测试 ---#
battery_type = "LCO"  # or "NMC", "LCO"
result_path = total_path
# Exclude files that contain 'discharge_wo_10' in their names
choice_data_file = [f for f in lco_files if 'wo_10' not in f]

test_files = [f for f in lco_files if 'LCO#4' in f]  
train_files = [f for f in lco_files if f not in test_files]
train_files.sort()
file_groups = {}
for file_path in train_files:
    # Extract filename from path
    filename = os.path.basename(file_path)
    
    # Extract battery name and state
    # Pattern: {battery_name}_{state}_wo_{number}.mat
    parts = filename.split('_wo_')[0]
    battery_name, state = parts.split('_', 1)
    
    # Create key for grouping
    group_key = (battery_name, state)
    
    # Add file to group
    if group_key not in file_groups:
        file_groups[group_key] = []
    file_groups[group_key].append(file_path)
    
#-----mixup data generation-----#    
# 定义保存路径
save_path = result_path
os.makedirs(save_path, exist_ok=True)
train_save_path = os.path.join(save_path, "train_data")
os.makedirs(train_save_path, exist_ok=True)
test_save_path = os.path.join(save_path, "test_data")
os.makedirs(test_save_path, exist_ok=True)

expand_factor = 1
# Generate weights with finer granularity based on expand_factor
step = 1 / (10 * expand_factor)  # smaller step size for higher granularity
weight = []
for i in range(1, 10 * expand_factor):  # more pairs based on expand_factor
    w1 = round(i * step, 3)  # round to 3 decimal places
    w2 = round(1 - w1, 3)
    weight.append((w1, w2))

for (battery, state), files in file_groups.items():
    files = sorted(files)
    if len(files) < 2:
        print(f"Not enough files for {battery}_{state} to perform mixup.")
        continue
    for idx in range(len(files)-1):
        
        stage1 = int(re.search(r'wo_(\d+)', files[idx]).group(1))
        stage2 = int(re.search(r'wo_(\d+)', files[idx+1]).group(1))
        
        tf1 = sio.loadmat(files[idx])
        filename1 = os.path.basename(files[idx])
        save_data(train_save_path, filename1, tf1)
        data1= tf1['data']
        mah1 = tf1['ah'][0][0]
        
        tf2 = sio.loadmat(files[idx+1])
        filename2 = os.path.basename(files[idx+1])
        save_data(train_save_path, filename2, tf2)
        data2 = tf2['data']
        mah2 = tf2['ah'][0][0]
        for wt1, wt2 in weight:
            new_data = wt1 * data1 + wt2 * data2
            new_mah = wt1 * mah1 + wt2 * mah2
            new_tf = {'data': new_data, 'ah': np.array([[new_mah]])}
            new_filename = f"{battery}_{state}_wo_{stage1}_w{wt1}_{stage2}_w{wt2}.mat"
            save_data(train_save_path, new_filename, new_tf)
# Save test data
for file_path in test_files:
    tf = sio.loadmat(file_path)
    filename = os.path.basename(file_path)
    save_data(test_save_path, filename, tf)
    


In [8]:
lco_json_dir = "/path/to/mae/data/lco_unknown"
os.makedirs(lco_json_dir, exist_ok=True)

trial_train_files = glob.glob(os.path.join(train_save_path,"*.mat"))
trial_test_files = glob.glob(os.path.join(test_save_path,"*.mat"))
split_data = {
    "LCO" : {
        'train': trial_train_files,
        'test': trial_test_files
    }
}
with open(os.path.join(lco_json_dir, f'LCO_unknown_#4.json'), 'w') as f:
    json.dump(split_data, f, indent=4)

In [ ]:
total_path = "/path/to/mae/data/lco_unknown/regreesion_data"
# --- 生成 LCO 增强与测试 ---#
battery_type = "LCO"  # or "NMC", "LCO"
sample_num = 60
# 随机选取8个LCO文件作为测试集，并且首先排除discharge_wo_10的数据
result_path = os.path.join(total_path, f"{battery_type}")

# Exclude files that contain 'discharge_wo_10' in their names
choice_data_file = [f for f in lco_files if 'wo_10' not in f]
# 生成sample_num组索引，每一组8个，然后按照索引获取测试集
# random.seed(42)  # For reproducibility
for i in range(sample_num):
    # 随机选择8个文件作为测试集
    # test_indices = random.sample(range(len(choice_data_file)), 8)
    # test_files = [choice_data_file[idx] for idx in test_indices]
    test_files = [f for f in test_files if 'LCO#2' in f]  
    train_files = [f for f in lco_files if f not in test_files]
    train_files.sort()
    file_groups = {}
    for file_path in train_files:
        # Extract filename from path
        filename = os.path.basename(file_path)
        
        # Extract battery name and state
        # Pattern: {battery_name}_{state}_wo_{number}.mat
        parts = filename.split('_wo_')[0]
        battery_name, state = parts.split('_', 1)
        
        # Create key for grouping
        group_key = (battery_name, state)
        
        # Add file to group
        if group_key not in file_groups:
            file_groups[group_key] = []
        file_groups[group_key].append(file_path)
        
    #-----mixup data generation-----#    
    # 定义保存路径
    save_path = os.path.join(result_path, f"trial_data_{i+1}")
    os.makedirs(save_path, exist_ok=True)
    train_save_path = os.path.join(save_path, "train_data")
    os.makedirs(train_save_path, exist_ok=True)
    test_save_path = os.path.join(save_path, "test_data")
    os.makedirs(test_save_path, exist_ok=True)
    
    expand_factor = 1
    # Generate weights with finer granularity based on expand_factor
    step = 1 / (10 * expand_factor)  # smaller step size for higher granularity
    weight = []
    for i in range(1, 10 * expand_factor):  # more pairs based on expand_factor
        w1 = round(i * step, 3)  # round to 3 decimal places
        w2 = round(1 - w1, 3)
        weight.append((w1, w2))

    for (battery, state), files in file_groups.items():
        files = sorted(files)
        if len(files) < 2:
            print(f"Not enough files for {battery}_{state} to perform mixup.")
            continue
        for idx in range(len(files)-1):
            
            stage1 = int(re.search(r'wo_(\d+)', files[idx]).group(1))
            stage2 = int(re.search(r'wo_(\d+)', files[idx+1]).group(1))
            
            tf1 = sio.loadmat(files[idx])
            filename1 = os.path.basename(files[idx])
            save_data(train_save_path, filename1, tf1)
            data1= tf1['data']
            mah1 = tf1['ah'][0][0]
            
            tf2 = sio.loadmat(files[idx+1])
            filename2 = os.path.basename(files[idx+1])
            save_data(train_save_path, filename2, tf2)
            data2 = tf2['data']
            mah2 = tf2['ah'][0][0]
            for wt1, wt2 in weight:
                new_data = wt1 * data1 + wt2 * data2
                new_mah = wt1 * mah1 + wt2 * mah2
                new_tf = {'data': new_data, 'ah': np.array([[new_mah]])}
                new_filename = f"{battery}_{state}_wo_{stage1}_w{wt1}_{stage2}_w{wt2}.mat"
                save_data(train_save_path, new_filename, new_tf)
    # Save test data
    for file_path in test_files:
        tf = sio.loadmat(file_path)
        filename = os.path.basename(file_path)
        save_data(test_save_path, filename, tf)
    


Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_charge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_charge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.
Not enough files for LCO#3_discharge to perform mixup.


In [ ]:
total_path = "/path/to/mae/data/final_test"
# --- 生成 NMC 增强与测试 ---#
battery_type = "NMC"  # or "NMC", "LCO"
sample_num = 60
# 随机选取8个LCO文件作为测试集，并且首先排除discharge_wo_10的数据
result_path = os.path.join(total_path, f"{battery_type}")

# Exclude files that contain 'discharge_wo_10' in their names
choice_data_file = [f for f in nmc_files if 'discharge_wo_10' not in f]
# 生成sample_num组索引，每一组8个，然后按照索引获取测试集
random.seed(42)
for i in range(sample_num):
    # 随机选择8个文件作为测试集
    test_indices = random.sample(range(len(choice_data_file)), 8)
    test_files = [choice_data_file[idx] for idx in test_indices]
    train_files = [f for f in nmc_files if f not in test_files]
    train_files.sort()
    file_groups = {}
    for file_path in train_files:
        # Extract filename from path
        filename = os.path.basename(file_path)
        
        # Extract battery name and state
        # Pattern: {battery_name}_{state}_wo_{number}.mat
        parts = filename.split('_wo_')[0]
        battery_name, state = parts.split('_', 1)
        
        # Create key for grouping
        group_key = (battery_name, state)
        
        # Add file to group
        if group_key not in file_groups:
            file_groups[group_key] = []
        file_groups[group_key].append(file_path)
        
    #-----mixup data generation-----#    
    # 定义保存路径
    save_path = os.path.join(result_path, f"trial_data_{i+1}")
    os.makedirs(save_path, exist_ok=True)
    train_save_path = os.path.join(save_path, "train_data")
    os.makedirs(train_save_path, exist_ok=True)
    test_save_path = os.path.join(save_path, "test_data")
    os.makedirs(test_save_path, exist_ok=True)
    
    expand_factor = 1
    # Generate weights with finer granularity based on expand_factor
    step = 1 / (10 * expand_factor)  # smaller step size for higher granularity
    weight = []
    for i in range(1, 10 * expand_factor):  # more pairs based on expand_factor
        w1 = round(i * step, 3)  # round to 3 decimal places
        w2 = round(1 - w1, 3)
        weight.append((w1, w2))

    for (battery, state), files in file_groups.items():
        files = sorted(files)
        if len(files) < 2:
            print(f"Not enough files for {battery}_{state} to perform mixup.")
            continue
        for idx in range(len(files)-1):
            
            stage1 = int(re.search(r'wo_(\d+)', files[idx]).group(1))
            stage2 = int(re.search(r'wo_(\d+)', files[idx+1]).group(1))
            
            tf1 = sio.loadmat(files[idx])
            filename1 = os.path.basename(files[idx])
            save_data(train_save_path, filename1, tf1)
            data1= tf1['data']
            mah1 = tf1['ah'][0][0]
            
            tf2 = sio.loadmat(files[idx+1])
            filename2 = os.path.basename(files[idx+1])
            save_data(train_save_path, filename2, tf2)
            data2 = tf2['data']
            mah2 = tf2['ah'][0][0]
            for wt1, wt2 in weight:
                new_data = wt1 * data1 + wt2 * data2
                new_mah = wt1 * mah1 + wt2 * mah2
                new_tf = {'data': new_data, 'ah': np.array([[new_mah]])}
                new_filename = f"{battery}_{state}_wo_{stage1}_w{wt1}_{stage2}_w{wt2}.mat"
                save_data(train_save_path, new_filename, new_tf)
    # Save test data
    for file_path in test_files:
        tf = sio.loadmat(file_path)
        filename = os.path.basename(file_path)
        save_data(test_save_path, filename, tf)
    


Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.


In [3]:
lco_train_path = os.path.join(total_path, "LCO")
trial_lco_files = os.listdir(lco_train_path)
trial_lco_files.sort()

nmc_train_path = os.path.join(total_path, "NMC")
trial_nmc_files = os.listdir(nmc_train_path)
trial_nmc_files.sort()

In [ ]:
total_path = "/path/to/mae/data/final_test"
lco_json_dir = os.path.join(total_path, "LCO_json")
os.makedirs(lco_json_dir, exist_ok=True)
for trial in trial_lco_files:
    trial_train_files = glob.glob(os.path.join(lco_train_path, trial, 'train_data',"*.mat"))
    trial_test_files = glob.glob(os.path.join(lco_train_path, trial, 'test_data',"*.mat"))
    split_data = {
        "LCO" : {
            'train': trial_train_files,
            'test': trial_test_files
        }
}

    with open(os.path.join(lco_json_dir, f'{trial}.json'), 'w') as f:
        json.dump(split_data, f, indent=4)

In [5]:
nmc_json_dir = os.path.join(total_path, "NMC_json")
os.makedirs(nmc_json_dir, exist_ok=True)
for trial in trial_nmc_files:
    trial_train_files = glob.glob(os.path.join(nmc_train_path, trial, 'train_data',"*.mat"))
    trial_test_files = glob.glob(os.path.join(nmc_train_path, trial, 'test_data',"*.mat"))
    split_data = {
        "NMC" : {
            'train': trial_train_files,
            'test': trial_test_files
        }
}


    with open(os.path.join(nmc_json_dir, f'{trial}.json'), 'w') as f:
        json.dump(split_data, f, indent=4)

In [ ]:

test_lco = [
    '/path/to/mae/data/mixup_data/original_data/LCO#2_charge_wo_4.mat',
	'/path/to/mae/data/mixup_data/original_data/LCO#3_charge_wo_4.mat',
 	'/path/to/mae/data/mixup_data/original_data/LCO#4_charge_wo_3.mat',
	'/path/to/mae/data/mixup_data/original_data/LCO#4_charge_wo_5.mat',
    '/path/to/mae/data/mixup_data/original_data/LCO#4_charge_wo_7.mat',
    '/path/to/mae/data/mixup_data/original_data/LCO#4_discharge_wo_11.mat',
    '/path/to/mae/data/mixup_data/original_data/LCO#4_discharge_wo_15.mat',
	'/path/to/mae/data/mixup_data/original_data/LCO#4_discharge_wo_16.mat',
]
# Sort the files for better readability
train_lco = sorted([f for f in lco_files if f not in test_lco])

In [9]:
test_nmc = [
    '/path/to/mae/data/mixup_data/original_data/NMC#1_discharge_wo_12.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#1_discharge_wo_15.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#2_charge_wo_3.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#2_charge_wo_6.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#2_discharge_wo_14.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#2_discharge_wo_16.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#3_discharge_wo_13.mat',
    '/path/to/mae/data/mixup_data/original_data/NMC#3_discharge_wo_16.mat',
]

train_nmc = sorted([f for f in nmc_files if f not in test_nmc])